# Diabetes Prediction: A Binary Classification Project

**Goal:** Predict whether a patient has diabetes from diagnostic measurements (Pima Indians Diabetes dataset, 768 patients, 8 features).

**What this project adds over a basic logistic-regression baseline**
1. Exploratory data analysis (class balance, distributions, correlations)
2. Fixing hidden missing values (zeros in glucose, BMI, etc. are impossible)
3. Leak-free preprocessing with a scikit-learn `Pipeline`
4. Comparing 5 models with stratified cross-validation
5. Hyperparameter tuning, evaluation with ROC-AUC / recall, and decision-threshold tuning
6. Feature importance, saving the model, and a prediction helper

> **Why recall matters:** in medical screening, missing a diabetic patient (false negative) is costlier than a false alarm. The baseline model's recall for diabetics is only 0.52, so we'll try to improve it.

## 1. Setup

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             accuracy_score, roc_auc_score, roc_curve, precision_recall_curve, recall_score)
from sklearn.inspection import permutation_importance

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load the data

In [ ]:
URL = "https://github.com/YBIFoundation/Dataset/raw/main/Diabetes.csv"
df = pd.read_csv(URL)          # if offline, download the CSV and pass its local path instead
print(df.shape)
df.head()

In [ ]:
df.info()
df.describe().T

## 3. Exploratory data analysis

### 3.1 Class balance

In [ ]:
counts = df["diabetes"].value_counts()
print(counts, "\n")
print((counts / len(df)).round(3))
ax = sns.countplot(x="diabetes", data=df)
ax.set_xticklabels(["No diabetes (0)", "Diabetes (1)"]); ax.set_title("Target distribution")
plt.show()

About a third of patients are diabetic, so the classes are **imbalanced**. Accuracy alone can be misleading, so we'll also track recall, F1 and ROC-AUC, and use *stratified* splits.

### 3.2 Hidden missing values

In [ ]:
zero_cols = ["glucose", "diastolic", "triceps", "insulin", "bmi"]
print("Count of zeros per column:")
print((df[zero_cols] == 0).sum())
print("\nShare of zeros (%):")
print(((df[zero_cols] == 0).mean() * 100).round(1))

Glucose, blood pressure, skin thickness, insulin and BMI **cannot physiologically be 0**, so these zeros are missing values recorded as 0. Insulin (~49%) and triceps (~30%) are affected the most. We convert them to `NaN` and impute inside the pipeline (so imputation statistics are learned from training data only).

In [ ]:
df_clean = df.copy()
df_clean[zero_cols] = df_clean[zero_cols].replace(0, np.nan)
df_clean.isna().sum()

### 3.3 Distributions by outcome

In [ ]:
features = [c for c in df_clean.columns if c != "diabetes"]
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.ravel(), features):
    sns.kdeplot(data=df_clean, x=col, hue="diabetes", common_norm=False, fill=True, ax=ax)
    ax.set_title(col)
plt.tight_layout(); plt.show()

### 3.4 Correlations

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df_clean.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix"); plt.show()
df_clean.corr()["diabetes"].drop("diabetes").sort_values(ascending=False)

Glucose is the strongest single predictor, followed by BMI, age and pregnancies. Age and pregnancies are correlated with each other, which is worth remembering when interpreting model coefficients.

## 4. Train / test split

In [ ]:
X = df_clean.drop(columns="diabetes")
y = df_clean["diabetes"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape)
print("Train positive rate:", round(y_train.mean(), 3), "| Test positive rate:", round(y_test.mean(), 3))

The test set is locked away until the final evaluation. All model selection uses cross-validation on the training set only.

## 5. Model comparison with cross-validation

In [ ]:
def make_pipe(model, scale=True):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scale: steps.append(("scale", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

models = {
    "Logistic Regression": make_pipe(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    "KNN":                 make_pipe(KNeighborsClassifier()),
    "SVM (RBF)":           make_pipe(SVC(probability=True, random_state=RANDOM_STATE)),
    "Random Forest":       make_pipe(RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE), scale=False),
    "Gradient Boosting":   make_pipe(GradientBoostingClassifier(random_state=RANDOM_STATE), scale=False),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "recall", "f1", "roc_auc"]

rows = []
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({"model": name, **{m: res[f"test_{m}"].mean() for m in scoring}})
results = pd.DataFrame(rows).set_index("model").round(3).sort_values("roc_auc", ascending=False)
results

In [ ]:
results.plot(kind="barh", figsize=(9, 5), colormap="viridis")
plt.title("5-fold CV performance (training set)"); plt.xlim(0.4, 0.9)
plt.legend(loc="lower right"); plt.show()

## 6. Hyperparameter tuning

We tune the two most promising model families. Scoring uses ROC-AUC, which is threshold-independent and robust to class imbalance.

In [ ]:
grids = {
    "Logistic Regression": (models["Logistic Regression"],
        {"model__C": [0.01, 0.1, 1, 10], "model__class_weight": [None, "balanced"]}),
    "Random Forest": (models["Random Forest"],
        {"model__max_depth": [3, 5, 8, None], "model__min_samples_leaf": [1, 3, 5],
         "model__class_weight": [None, "balanced"]}),
}
best = {}
for name, (pipe, grid) in grids.items():
    gs = GridSearchCV(pipe, grid, cv=cv, scoring="roc_auc", n_jobs=-1).fit(X_train, y_train)
    best[name] = gs.best_estimator_
    print(f"{name}: CV ROC-AUC = {gs.best_score_:.3f} | best params = {gs.best_params_}")

## 7. Final evaluation on the held-out test set

In [ ]:
baseline = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
# Baseline: zeros kept as-is (like the original notebook), for fair before/after comparison
X_train_raw, X_test_raw = df.loc[X_train.index].drop(columns="diabetes"), df.loc[X_test.index].drop(columns="diabetes")
baseline.fit(X_train_raw, y_train)

candidates = {"Baseline LR (raw zeros)": (baseline, X_test_raw), **{k: (v, X_test) for k, v in best.items()}}
rows = []
for name, (m, Xt) in candidates.items():
    pred, proba = m.predict(Xt), m.predict_proba(Xt)[:, 1]
    rows.append({"model": name, "accuracy": accuracy_score(y_test, pred),
                 "recall": recall_score(y_test, pred), "roc_auc": roc_auc_score(y_test, proba)})
pd.DataFrame(rows).set_index("model").round(3)

In [ ]:
# Choose the best tuned model by CV score
final_name = max(best, key=lambda n: cross_validate(best[n], X_train, y_train, cv=cv, scoring="roc_auc")["test_score"].mean())
final_model = best[final_name]
print("Selected model:", final_name)

proba = final_model.predict_proba(X_test)[:, 1]
pred = final_model.predict(X_test)
print(classification_report(y_test, pred, target_names=["No diabetes", "Diabetes"]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred), display_labels=["No", "Yes"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion matrix (threshold 0.5)")
fpr, tpr, _ = roc_curve(y_test, proba)
axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, proba):.3f}"); axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set(xlabel="False positive rate", ylabel="True positive rate", title="ROC curve"); axes[1].legend()
plt.tight_layout(); plt.show()

## 8. Decision-threshold tuning

The default 0.5 cutoff isn't optimal for screening. Here we pick the threshold using **cross-validated predictions on the training set** (not the test set) to reach a target recall, then check the trade-off on the test set.

In [ ]:
from sklearn.model_selection import cross_val_predict
oof = cross_val_predict(final_model, X_train, y_train, cv=cv, method="predict_proba")[:, 1]
prec, rec, thr = precision_recall_curve(y_train, oof)

TARGET_RECALL = 0.80
ok = np.where(rec[:-1] >= TARGET_RECALL)[0]
threshold = thr[ok[-1]] if len(ok) else 0.5     # highest threshold that still meets target recall
print(f"Chosen threshold: {threshold:.2f}")

pred_t = (proba >= threshold).astype(int)
print(classification_report(y_test, pred_t, target_names=["No diabetes", "Diabetes"]))

plt.figure(figsize=(7, 4.5))
plt.plot(thr, prec[:-1], label="Precision"); plt.plot(thr, rec[:-1], label="Recall")
plt.axvline(threshold, color="gray", ls="--", label=f"Chosen = {threshold:.2f}")
plt.xlabel("Threshold"); plt.title("Precision / recall trade-off (CV predictions)"); plt.legend(); plt.show()

Lowering the threshold catches more diabetic patients (higher recall) at the cost of more false alarms (lower precision). The right balance is a clinical decision.

## 9. What drives the predictions?

In [ ]:
imp = permutation_importance(final_model, X_test, y_test, scoring="roc_auc", n_repeats=30, random_state=RANDOM_STATE)
imp_df = pd.Series(imp.importances_mean, index=X.columns).sort_values()
imp_df.plot(kind="barh", figsize=(8, 4.5), xerr=pd.Series(imp.importances_std, index=X.columns)[imp_df.index])
plt.title("Permutation importance (drop in test ROC-AUC)"); plt.show()

## 10. Save the model and make predictions

In [ ]:
joblib.dump({"model": final_model, "threshold": float(threshold), "features": list(X.columns)}, "diabetes_model.joblib")

def predict_patient(pregnancies, glucose, diastolic, triceps, insulin, bmi, dpf, age, bundle_path="diabetes_model.joblib"):
    """Return (probability, label). Pass None for any unknown measurement."""
    b = joblib.load(bundle_path)
    row = pd.DataFrame([[pregnancies, glucose, diastolic, triceps, insulin, bmi, dpf, age]], columns=b["features"]).astype(float)
    p = b["model"].predict_proba(row)[0, 1]
    return round(float(p), 3), "Diabetes likely" if p >= b["threshold"] else "Diabetes unlikely"

print(predict_patient(6, 148, 72, 35, None, 33.6, 0.627, 50))
print(predict_patient(1, 89, 66, 23, 94, 28.1, 0.167, 21))

## 11. Conclusions

*(Fill in with your own numbers after running the notebook; these are the points to check.)*

- **Data quality:** compare the baseline (raw zeros) with the tuned pipeline in Section 7. Did proper handling of impossible zeros help?
- **Model choice:** on small tabular data, tuned logistic regression is often on par with tree ensembles and stays interpretable. Check Section 5 to see if that holds here.
- **Accuracy hides the real problem:** look at recall for the diabetic class before and after threshold tuning (Section 8), and note the precision you give up.
- **Feature drivers:** Section 9 shows which features matter most (glucose is expected to lead).

**Limitations & next steps**
- Only 768 rows, so the test set is small and metrics are noisy; report confidence intervals or repeat the CV.
- The dataset covers a single population (Pima Indian women aged 21+), so results may not generalise.
- Try KNN/iterative imputation, feature engineering (e.g. glucose×BMI), SHAP explanations, and calibration.
- Wrap `predict_patient` in a Streamlit or Flask app.

*This is an educational project, not a medical diagnostic tool.*